# Hybrid — SASRec + audio-content two-tower on Yambda-50M

Effective item embedding `v_j = e_j + alpha * mask_j * ContentMLP(audio_j)` fused end-to-end. `alpha` is learned: alpha->0 means pure SASRec, so its final value shows how much the audio content helps. User side is the SASRec sequence rep.

**Bars (our harness):** ItemKNN-bm25 0.0709, **SASRec 0.0735**. Goal: beat SASRec.

**Before running:** add the content-embeddings dataset via *Add Data* → your `yambda-50m-content-emb` (the output of `03_content_emb_prep`). Accelerator → **GPU**, Internet On.

In [ ]:
!pip install -q --no-cache-dir --upgrade "git+https://github.com/ak1232320/nndl_capstone_2026.git"

In [ ]:
import os, glob, numpy as np, torch
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

# Find the content-embeddings dataset regardless of its exact slug.
emb_file = glob.glob("/kaggle/input/**/content_emb.npy", recursive=True)[0]
base = os.path.dirname(emb_file)
content_emb = np.load(f"{base}/content_emb.npy")
content_mask = np.load(f"{base}/content_mask.npy")
saved_item_ids = np.load(f"{base}/item_ids.npy")
print("content:", content_emb.shape, content_emb.dtype, "coverage", content_mask.mean().round(3), "from", base)

In [ ]:
from ymrec.data.sequences import build_sequences

data = build_sequences(size="50m", maxlen=200)
# The content rows must line up with the rebuilt vocab (both are deterministic from the GTS split).
assert np.array_equal(saved_item_ids, data.item_ids), "vocab mismatch — re-run 03 with the same size/maxlen"
print(f"aligned OK | items={data.n_items:,} eval_users={len(data.eval_pos):,}")

In [ ]:
import time
from ymrec.models.hybrid import train_and_eval

t0 = time.time()
model, best = train_and_eval(
    data, content_emb, content_mask,
    d=64, n_blocks=2, n_heads=1, dropout=0.2,
    epochs=120, batch_size=128, lr=1e-3, eval_every=10,
)
print(f"\ntrained in {(time.time()-t0)/60:.1f} min | learned alpha={float(model.alpha):.3f} | best: {best}")

In [ ]:
import pandas as pd
ref = {"MostPop": 0.0171, "ItemKNN-bm25": 0.0709, "SASRec": 0.0735,
       "Hybrid (ours)": round(best["ndcg@10"], 4)}
pd.Series(ref, name="NDCG@10").sort_values(ascending=False).to_frame()